In [2]:
import mediapipe as mp
print(mp.__version__)

0.10.21


In [3]:
import os
import cv2
import numpy as np
import mediapipe as mp
import pickle
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import random
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")
print(f"OpenCV version: {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
print(f"NumPy version: {np.__version__}")

All libraries imported successfully.
OpenCV version: 4.11.0
MediaPipe version: 0.10.21
NumPy version: 1.26.4


In [4]:
DATASET_ROOT = "/home/user/22059240/FYP/data/URFD"

FALL_DIR = os.path.join(DATASET_ROOT, "fall")
ADL_DIR = os.path.join(DATASET_ROOT, "adl")

OUTPUT_DIR = "/home/user/22059240/FYP/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pipeline parameters
TARGET_FPS      = 30        # Resample all videos to 30 fps
TARGET_WIDTH    = 640       # Resize width
TARGET_HEIGHT   = 480       # Resize height
WINDOW_SIZE     = 30        # Sliding window — 30 frames per sequence
STEP_TRAIN      = 5         # Step size during training window extraction
STEP_INFER      = 1         # Step size during inference (not used here, for reference)
FEATURE_DIM     = 166       # 132 keypoints + 33 velocity + 1 aspect ratio
VISIBILITY_THR  = 0.5       # Minimum visibility score to count keypoint as detected
MAX_ZERO_RATIO  = 0.20      # Discard sequence if >20% frames have no detection
TEST_SIZE       = 0.20      # 80/20 train-test split
RANDOM_SEED     = 42

# Labels
LABEL_FALL    = 1
LABEL_NONFAIL = 0

print("Configuration set.")
print(f"  Dataset root : {DATASET_ROOT}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  Window size  : {WINDOW_SIZE} frames")
print(f"  Feature dim  : {FEATURE_DIM}")

Configuration set.
  Dataset root : /home/user/22059240/FYP/data/URFD
  Output dir   : /home/user/22059240/FYP/processed
  Window size  : 30 frames
  Feature dim  : 166


In [5]:
def get_sequences(directory):
    """Return sorted list of image-sequence folder paths."""
    sequences = [
        os.path.join(directory, f)
        for f in sorted(os.listdir(directory))
        if os.path.isdir(os.path.join(directory, f))
    ]
    return sequences


fall_sequences = get_sequences(FALL_DIR)
adl_sequences  = get_sequences(ADL_DIR)

print(f"Fall sequences found : {len(fall_sequences)}")
print(f"ADL sequences found  : {len(adl_sequences)}")
print(f"Total                : {len(fall_sequences) + len(adl_sequences)}")

# Quick sanity check — show first 3 of each
print("\nSample fall sequences:")
for seq in fall_sequences[:3]:
    print(f"  {os.path.basename(seq)}")

print("\nSample ADL sequences:")
for seq in adl_sequences[:3]:
    print(f"  {os.path.basename(seq)}")

Fall sequences found : 30
ADL sequences found  : 39
Total                : 69

Sample fall sequences:
  fall-01-cam0-rgb
  fall-02-cam0-rgb
  fall-03-cam0-rgb

Sample ADL sequences:
  adl-01-cam0-rgb
  adl-02-cam0-rgb
  adl-03-cam0-rgb


In [6]:
def augment_frame(frame):
    """
    Apply one random augmentation to a single BGR frame.
    Returns a list of augmented frames (original + augmented versions).
    Augmentations (Section 3.5.4):
      1. Horizontal flip
      2. Random rotation ±10°
      3. Brightness variation ±20%
    """
    augmented = []

    # 1. Horizontal flip
    flipped = cv2.flip(frame, 1)
    augmented.append(flipped)

    # 2. Random rotation ±10°
    h, w = frame.shape[:2]
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(frame, M, (w, h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT_101)
    augmented.append(rotated)

    # 3. Brightness variation ×[0.8, 1.2]
    factor = random.uniform(0.8, 1.2)
    bright = cv2.convertScaleAbs(frame, alpha=factor, beta=0)
    augmented.append(bright)

    return augmented   # Returns 3 augmented versions


print("Augmentation functions defined.")

Augmentation functions defined.


In [7]:
import mediapipe as mp

print(mp.__version__)
print(mp)

0.10.21
<module 'mediapipe' from '/home/user/22059240/fyp-env/lib/python3.12/site-packages/mediapipe/__init__.py'>


In [ ]:
# Initialise MediaPipe BlazePose
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose_detector = mp_pose.Pose(
    static_image_mode=False,        # Video mode — uses tracking between frames
    model_complexity=1,             # 0=lite, 1=full, 2=heavy
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

NUM_KEYPOINTS  = 33
COORDS_PER_KP  = 4    # x, y, z, visibility


def extract_keypoints(frame_rgb):
    """
    Run BlazePose on one RGB frame.
    Returns:
        kp_vector (np.ndarray): shape (132,) — flattened [x,y,z,vis] * 33
        detected  (bool)      : True if at least one person was detected
    """
    results = pose_detector.process(frame_rgb)

    if results.pose_landmarks is None:
        return np.zeros(NUM_KEYPOINTS * COORDS_PER_KP), False

    kp_vector = []
    for lm in results.pose_landmarks.landmark:
        kp_vector.extend([lm.x, lm.y, lm.z, lm.visibility])

    return np.array(kp_vector, dtype=np.float32), True


def normalize_keypoints(kp_vector):
    """
    Person-relative bounding box normalisation (Section 3.5.2).
    Re-scales x and y coordinates to [0,1] relative to the
    bounding box of all visible keypoints, rather than the full image.
    z and visibility are left unchanged.
    """
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP).copy()

    # Only consider keypoints with visibility >= threshold
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return kp.flatten()

    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    scale = max(x_max - x_min, y_max - y_min, 1e-6)   # one shared scale for x and y

    kp[:, 0] = (kp[:, 0] - x_min) / scale
    kp[:, 1] = (kp[:, 1] - y_min) / scale
    # z and visibility untouched

    return kp.flatten()


def compute_aspect_ratio(kp_vector):
    """
    Body bounding box aspect ratio = height / width (Section 3.5.3).
    Returns a single float. Returns 1.0 if bbox is degenerate.
    """
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP)
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return 1.0

    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    width  = xs.max() - xs.min()
    height = ys.max() - ys.min()

    if width < 1e-6:
        return 1.0
    return float(min(height / width, 5.0))


print("BlazePose initialised and feature functions defined.")

BlazePose initialised and feature functions defined.


I0000 00:00:1790863845.612179  175261 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1790863845.742860  178470 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 570.172.08), renderer: NVIDIA L4/PCIe/SSE2


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1790863845.863889  178462 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790863845.936884  178459 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [9]:
def extract_features_from_video(video_path, is_training=True):
    """
    Process one video file end-to-end:
      1. Read and resample to TARGET_FPS
      2. Resize frames to TARGET_WIDTH x TARGET_HEIGHT
      3. Optionally augment (training only)
      4. Extract BlazePose keypoints per frame
      5. Normalise coordinates
      6. Compute vertical velocity per keypoint
      7. Compute body aspect ratio per frame
      8. Return list of 166-dim feature vectors (one per frame)
         and a flag indicating whether this video should be kept.

    Returns:
        feature_matrix (np.ndarray | None): shape (n_frames, 166)
        keep (bool): False if too many undetected frames
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  [WARNING] Cannot open: {video_path}")
        return None, False

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps <= 0:
        original_fps = TARGET_FPS

    # Frame sampling interval to achieve TARGET_FPS
    sample_interval = max(1, round(original_fps / TARGET_FPS))

    raw_frames = []
    frame_idx  = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_interval == 0:
            # Resize to target resolution
            frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT))
            raw_frames.append(frame)
        frame_idx += 1

    cap.release()

    if len(raw_frames) == 0:
        return None, False

    # ── Augmentation (training only) ──────────────────────────
    # We process the original frames here.
    # Augmented versions are handled separately in Cell 7.
    frames_to_process = raw_frames

    # ── Keypoint extraction ────────────────────────────────────
    kp_vectors  = []   # shape per entry: (132,)
    detected_flags = []

    for frame_bgr in frames_to_process:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        kp_raw, detected = extract_keypoints(frame_rgb)

        if detected:
            kp_norm = normalize_keypoints(kp_raw)
        else:
            kp_norm = np.zeros(NUM_KEYPOINTS * COORDS_PER_KP, dtype=np.float32)

        kp_vectors.append(kp_norm)
        detected_flags.append(detected)

    # ── Check zero-frame ratio ─────────────────────────────────
    zero_ratio = 1.0 - (sum(detected_flags) / len(detected_flags))
    if zero_ratio > MAX_ZERO_RATIO:
        return None, False   # Discard this video

    # ── Vertical velocity (Section 3.5.3) ─────────────────────
    # Δy for each of the 33 keypoints between consecutive frames
    # y is at index 1 of every [x, y, z, vis] group, i.e., indices [1, 5, 9, ...]
    y_indices = [i * COORDS_PER_KP + 1 for i in range(NUM_KEYPOINTS)]

    velocities = []
    for i, kp in enumerate(kp_vectors):
        if i == 0:
            vel = np.zeros(NUM_KEYPOINTS, dtype=np.float32)
        else:
            prev_y = kp_vectors[i - 1][y_indices]
            curr_y = kp[y_indices]
            vel    = (curr_y - prev_y).astype(np.float32)
        velocities.append(vel)

    # ── Body aspect ratio (Section 3.5.3) ─────────────────────
    aspect_ratios = [
        compute_aspect_ratio(kp) for kp in kp_vectors
    ]

    # ── Assemble 166-dim feature vectors ──────────────────────
    feature_matrix = []
    for kp, vel, ar in zip(kp_vectors, velocities, aspect_ratios):
        fv = np.concatenate([kp, vel, [ar]]).astype(np.float32)
        feature_matrix.append(fv)

    feature_matrix = np.array(feature_matrix, dtype=np.float32)
    # shape: (n_frames, 166)

    return feature_matrix, True


print("Per-video feature extraction function defined.")

Per-video feature extraction function defined.


In [10]:
def extract_windows(feature_matrix, label, step=STEP_TRAIN):
    """
    Apply sliding window to a feature matrix (Section 3.5.5).
    
    Args:
        feature_matrix : np.ndarray shape (n_frames, 166)
        label          : int — 1 for fall, 0 for non-fall
        step           : int — window step size

    Returns:
        windows : np.ndarray shape (n_windows, 30, 166)
        labels  : np.ndarray shape (n_windows,)
    """
    n_frames = len(feature_matrix)
    windows  = []
    labels   = []

    for start in range(0, n_frames - WINDOW_SIZE + 1, step):
        window = feature_matrix[start : start + WINDOW_SIZE]
        if len(window) == WINDOW_SIZE:
            windows.append(window)
            labels.append(label)

    if len(windows) == 0:
        return np.empty((0, WINDOW_SIZE, FEATURE_DIM), dtype=np.float32), np.array([], dtype=np.int32)

    return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.int32)


print("Sliding window function defined.")

Sliding window function defined.


In [11]:
def process_dataset(sequence_paths, label, is_training=True, step=STEP_TRAIN):
    """
    Process URFD image-sequence folders and return sliding windows and labels.
    For training sequences, also generates augmented versions.
    """
    all_windows = []
    all_labels = []
    skipped = 0

    for seq_path in tqdm(
        sequence_paths,
        desc=f"Processing {'fall' if label == 1 else 'ADL'} sequences"
    ):

        # Get PNG frames and sort by frame number
        frame_files = [
            os.path.join(seq_path, f)
            for f in os.listdir(seq_path)
            if f.lower().endswith(".png")
        ]

        frame_files.sort(
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("-")[-1])
        )

        raw_frames = []

        for frame_path in frame_files:
            frame = cv2.imread(frame_path)

            if frame is None:
                continue

            frame = cv2.resize(
                frame,
                (TARGET_WIDTH, TARGET_HEIGHT)
            )

            raw_frames.append(frame)

        if len(raw_frames) == 0:
            skipped += 1
            continue

        # ── Process original sequence ─────────────────────────
        kp_vecs = []
        det_flags = []

        for frame_bgr in raw_frames:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            kp_raw, detected = extract_keypoints(frame_rgb)

            if detected:
                kp_norm = normalize_keypoints(kp_raw)
            else:
                kp_norm = np.zeros(
                    NUM_KEYPOINTS * COORDS_PER_KP,
                    dtype=np.float32
                )

            kp_vecs.append(kp_norm)
            det_flags.append(detected)

        zero_ratio = 1.0 - (sum(det_flags) / len(det_flags))

        if zero_ratio > MAX_ZERO_RATIO:
            skipped += 1
            continue

        # Vertical velocity
        y_indices = [
            i * COORDS_PER_KP + 1
            for i in range(NUM_KEYPOINTS)
        ]

        vels = []

        for i, kp in enumerate(kp_vecs):
            if i == 0:
                vels.append(
                    np.zeros(NUM_KEYPOINTS, dtype=np.float32)
                )
            else:
                vels.append(
                    (
                        kp_vecs[i][y_indices]
                        - kp_vecs[i - 1][y_indices]
                    ).astype(np.float32)
                )

        # Body aspect ratio
        ars = [
            compute_aspect_ratio(kp)
            for kp in kp_vecs
        ]

        feat_matrix = np.array([
            np.concatenate([kp, vel, [ar]]).astype(np.float32)
            for kp, vel, ar in zip(kp_vecs, vels, ars)
        ], dtype=np.float32)

        wins, labs = extract_windows(
            feat_matrix,
            label,
            step=step
        )

        if len(wins) > 0:
            all_windows.append(wins)
            all_labels.append(labs)

        # ── Augmentation (training only) ──────────────────────
        if is_training:

            aug_frame_sets = [[] for _ in range(3)]

            for frame in raw_frames:
                aug_versions = augment_frame(frame)

                for aug_idx, aug_frame in enumerate(aug_versions):
                    aug_frame_sets[aug_idx].append(aug_frame)

            for aug_frames in aug_frame_sets:

                kp_vecs = []
                det_flags = []

                for frame_bgr in aug_frames:

                    frame_rgb = cv2.cvtColor(
                        frame_bgr,
                        cv2.COLOR_BGR2RGB
                    )

                    kp_raw, detected = extract_keypoints(
                        frame_rgb
                    )

                    if detected:
                        kp_norm = normalize_keypoints(kp_raw)
                    else:
                        kp_norm = np.zeros(
                            NUM_KEYPOINTS * COORDS_PER_KP,
                            dtype=np.float32
                        )

                    kp_vecs.append(kp_norm)
                    det_flags.append(detected)

                zero_ratio = 1.0 - (
                    sum(det_flags) / len(det_flags)
                )

                if zero_ratio > MAX_ZERO_RATIO:
                    continue

                vels = []

                for i, kp in enumerate(kp_vecs):
                    if i == 0:
                        vels.append(
                            np.zeros(
                                NUM_KEYPOINTS,
                                dtype=np.float32
                            )
                        )
                    else:
                        vels.append(
                            (
                                kp_vecs[i][y_indices]
                                - kp_vecs[i - 1][y_indices]
                            ).astype(np.float32)
                        )

                ars = [
                    compute_aspect_ratio(kp)
                    for kp in kp_vecs
                ]

                aug_feat = np.array([
                    np.concatenate(
                        [kp, vel, [ar]]
                    ).astype(np.float32)
                    for kp, vel, ar
                    in zip(kp_vecs, vels, ars)
                ], dtype=np.float32)

                aug_wins, aug_labs = extract_windows(
                    aug_feat,
                    label,
                    step=step
                )

                if len(aug_wins) > 0:
                    all_windows.append(aug_wins)
                    all_labels.append(aug_labs)

    print(
        f"  Skipped (too many missing detections): {skipped}"
    )

    if len(all_windows) == 0:
        return (
            np.empty(
                (0, WINDOW_SIZE, FEATURE_DIM),
                dtype=np.float32
            ),
            np.array([], dtype=np.int32)
        )

    X = np.concatenate(
        all_windows,
        axis=0
    )

    y = np.concatenate(
        all_labels,
        axis=0
    )

    return X, y


print("Dataset processing function defined.")

Dataset processing function defined.


In [12]:
# ─────────────────────────────────────────────────────────────
# IMPORTANT: Split at SEQUENCE level before any feature extraction
# This prevents data leakage from sliding window overlaps
# ─────────────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Fall sequences split
fall_train_seqs, fall_test_seqs = train_test_split(
    fall_sequences,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

# ADL sequences split
adl_train_seqs, adl_test_seqs = train_test_split(
    adl_sequences,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

print(f"Fall — train: {len(fall_train_seqs)} | test: {len(fall_test_seqs)}")
print(f"ADL  — train: {len(adl_train_seqs)}  | test: {len(adl_test_seqs)}")
print(f"Total train sequences : {len(fall_train_seqs) + len(adl_train_seqs)}")
print(f"Total test sequences  : {len(fall_test_seqs) + len(adl_test_seqs)}")

Fall — train: 24 | test: 6
ADL  — train: 31  | test: 8
Total train sequences : 55
Total test sequences  : 14


In [13]:
print("=" * 55)
print("Processing TRAINING set...")
print("=" * 55)

# Fall training sequences
print("\n[FALL - Train]")
X_fall_train, y_fall_train = process_dataset(
    fall_train_seqs,
    label=LABEL_FALL,
    is_training=True,
    step=STEP_TRAIN
)

# ADL training sequences
print("\n[ADL - Train]")
X_adl_train, y_adl_train = process_dataset(
    adl_train_seqs,
    label=LABEL_NONFAIL,
    is_training=True,
    step=STEP_TRAIN
)

# Combine and shuffle
X_train = np.concatenate(
    [X_fall_train, X_adl_train],
    axis=0
)

y_train = np.concatenate(
    [y_fall_train, y_adl_train],
    axis=0
)

# Shuffle
shuffle_idx = np.random.permutation(len(X_train))

X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print(f"\nTraining set shape  : {X_train.shape}")
print(f"Training labels     : {y_train.shape}")
print(f"Fall windows        : {y_train.sum()}")
print(f"Non-fall windows    : {(y_train == 0).sum()}")

Processing TRAINING set...

[FALL - Train]


Processing fall sequences: 100%|██████████| 24/24 [03:22<00:00,  8.45s/it]


  Skipped (too many missing detections): 5

[ADL - Train]


Processing ADL sequences: 100%|██████████| 31/31 [07:34<00:00, 14.67s/it]

  Skipped (too many missing detections): 16

Training set shape  : (3123, 30, 166)
Training labels     : (3123,)
Fall windows        : 1179
Non-fall windows    : 1944


In [14]:
print("=" * 55)
print("Processing TEST set (no augmentation)...")
print("=" * 55)

# Fall test sequences
print("\n[FALL - Test]")
X_fall_test, y_fall_test = process_dataset(
    fall_test_seqs,
    label=LABEL_FALL,
    is_training=False,
    step=STEP_TRAIN
)

# ADL test sequences
print("\n[ADL - Test]")
X_adl_test, y_adl_test = process_dataset(
    adl_test_seqs,
    label=LABEL_NONFAIL,
    is_training=False,
    step=STEP_TRAIN
)

# Combine
X_test = np.concatenate(
    [X_fall_test, X_adl_test],
    axis=0
)

y_test = np.concatenate(
    [y_fall_test, y_adl_test],
    axis=0
)

# Shuffle test set
shuffle_idx = np.random.permutation(len(X_test))

X_test = X_test[shuffle_idx]
y_test = y_test[shuffle_idx]

print(f"\nTest set shape  : {X_test.shape}")
print(f"Test labels     : {y_test.shape}")
print(f"Fall windows    : {y_test.sum()}")
print(f"Non-fall windows: {(y_test == 0).sum()}")

Processing TEST set (no augmentation)...

[FALL - Test]


Processing fall sequences: 100%|██████████| 6/6 [00:23<00:00,  3.89s/it]


  Skipped (too many missing detections): 2

[ADL - Test]


Processing ADL sequences: 100%|██████████| 8/8 [01:15<00:00,  9.43s/it]

  Skipped (too many missing detections): 2

Test set shape  : (243, 30, 166)
Test labels     : (243,)
Fall windows    : 53
Non-fall windows: 190


In [15]:
# Class weights to handle imbalance (Section 3.5.6)
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))

print("Class weights (for training):")
for cls, w in class_weight_dict.items():
    label_name = "Fall" if cls == 1 else "Non-fall"
    print(f"  Class {cls} ({label_name}): {w:.4f}")

Class weights (for training):
  Class 0 (Non-fall): 0.8032
  Class 1 (Fall): 1.3244


In [16]:
# Save everything to disk so you don't have to rerun extraction
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"),  y_test)

with open(os.path.join(OUTPUT_DIR, "class_weights.pkl"), "wb") as f:
    pickle.dump(class_weight_dict, f)

print("Saved:")
print(f"  {OUTPUT_DIR}/X_train.npy  —  {X_train.shape}")
print(f"  {OUTPUT_DIR}/y_train.npy  —  {y_train.shape}")
print(f"  {OUTPUT_DIR}/X_test.npy   —  {X_test.shape}")
print(f"  {OUTPUT_DIR}/y_test.npy   —  {y_test.shape}")
print(f"  {OUTPUT_DIR}/class_weights.pkl")

Saved:
  /home/user/22059240/FYP/processed/X_train.npy  —  (3123, 30, 166)
  /home/user/22059240/FYP/processed/y_train.npy  —  (3123,)
  /home/user/22059240/FYP/processed/X_test.npy   —  (243, 30, 166)
  /home/user/22059240/FYP/processed/y_test.npy   —  (243,)
  /home/user/22059240/FYP/processed/class_weights.pkl
